# DFD-HR 分层路由方法教程

这个 Notebook 从代码而不是从口号出发，追踪 DFD-HR 的数据流、张量形状和训练信号。它不加载权重、不读取数据集、不使用 GPU，也不会启动训练。

主要参考：[DFD-HR (CVPR 2026)](https://openaccess.thecvf.com/content/CVPR2026/papers/Sun_DFD-HR_Generalizable_Deepfake_Detection_via_Hierarchical_Routing_Learning_CVPR_2026_paper.pdf)、[CLIP](https://arxiv.org/abs/2103.00020)、[Adapter](https://arxiv.org/abs/1902.00751) 和 [Switch Transformer / MoE](https://arxiv.org/abs/2101.03961)。

## 1. 先区分技术来源

| 标记 | 含义 |
| --- | --- |
| 成熟技术 | 已有广泛文献和工程实践，例如 ViT、CLIP、Attention、Adapter、MoE、交叉熵。 |
| DFD-HR 贡献 | 论文明确提出的组合或约束：Early Layer Pruning、Token Selection + Spearman、Expert Routing 与多尺度融合。 |
| 研究方向 | 从当前实现观察到、值得做消融的假设，不代表已经证明有效或具有新颖性。 |

最重要的阅读原则：**路由不改变特征宽度，它改变哪些样本、哪些 token、哪些专家参与计算。**

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PaperSpec:
    input_size: int = 448
    crop_size: int = 224
    patch_size: int = 14
    hidden_dim: int = 1024
    projection_dim: int = 768
    layers: int = 24
    remain_layer: int = 20
    capacity: float = 0.75
    experts: int = 4
    top_k: int = 4


spec = PaperSpec()
patches_per_side = spec.crop_size // spec.patch_size
tokens_per_crop = patches_per_side**2 + 1
routed_layers = list(range(spec.remain_layer, spec.layers))
selected_tokens = max(1, int(spec.capacity * tokens_per_crop))

print("patch grid:", (patches_per_side, patches_per_side))
print("tokens per crop (CLS included):", tokens_per_crop)
print("routed layer indices:", routed_layers)
print("selected / bypassed tokens:", selected_tokens, "/", tokens_per_crop - selected_tokens)


## 2. 多尺度输入与特征矩阵

`forward_query_loss` 接收 `[B, 3, 448, 448]`：

1. 全局分支缩放到 224，得到 `[B, 3, 224, 224]`。
2. 局部分支保留 448，再切成 2 x 2 棋盘，得到 `[4B, 3, 224, 224]`。
3. 每个 crop 经过 CLIP ViT-L/14，得到 1 个 CLS 和 256 个 patch token，即 `[N, 257, 1024]`。
4. CLIP 视觉投影把最后一维从 1024 变为 768，但 token 数不变。
5. 一个可学习 query 对四个局部 CLS 做多头注意力，输出 `[B, 1, 768]`。
6. 全局 CLS 与融合后的局部 CLS 拼接为 `[B, 1536]`，送入二分类头。

成熟技术是 CLIP、视觉投影和多头注意力；DFD-HR 使用全局语义作为局部 token 排序先验，并把多尺度特征接入同一套路由网络。

In [ ]:
batch_size = 8
shape_trace = {
    "input": (batch_size, 3, 448, 448),
    "global_crops": (batch_size, 3, 224, 224),
    "local_crops": (4 * batch_size, 3, 224, 224),
    "global_hidden": (batch_size, tokens_per_crop, spec.hidden_dim),
    "local_hidden": (4 * batch_size, tokens_per_crop, spec.hidden_dim),
    "global_projected": (batch_size, tokens_per_crop, spec.projection_dim),
    "local_projected": (4 * batch_size, tokens_per_crop, spec.projection_dim),
    "local_query_fusion": (batch_size, 1, spec.projection_dim),
    "classifier_feature": (batch_size, 2 * spec.projection_dim),
    "logits": (batch_size, 2),
}

for name, shape in shape_trace.items():
    print(f"{name:22s} {shape}")

assert shape_trace["classifier_feature"][-1] == 1536
assert shape_trace["logits"] == (batch_size, 2)


## 3. 层路由：决定样本是否继续深入

`features_encoder` 的前 20 个 block 对所有样本执行。第 20-23 个 block 的 `LayerRouter` 把 CLS `[B, 1024]` 映射为继续计算的 logit `[B, 1]`。

- 训练：Gumbel-Sigmoid + straight-through estimator，前向是硬 0/1，反向仍可传梯度。
- 推理：`sigmoid(logit) > 0.5`。
- `active_mask = active_mask * decision`，所以退出不可逆；已经退出的样本不会在更深层重新进入。
- 退出样本保留退出时的 token，最后仍可参与分类。

Gumbel 和 straight-through 是成熟估计技术；按样本学习不同 CLIP 前向深度是 DFD-HR 的 Early Layer Pruning。

In [ ]:
# 仅演示不可逆掩码，不模拟真实 router 数值。
active = [1, 1, 1, 1]
decisions = {
    20: [1, 0, 1, 1],
    21: [1, 1, 0, 1],
    22: [0, 1, 1, 1],
    23: [1, 1, 1, 1],
}

for layer, decision in decisions.items():
    active = [old * new for old, new in zip(active, decision)]
    print(f"after layer {layer}: {active}")

assert active == [0, 0, 0, 1]


## 4. Token Selection：决定一个 block 处理哪些 token

对仍活跃的样本，`TokenRouter` 产生 `[B_active, 257]` 分数。默认 capacity=0.75，因此每个 crop 选 192 个 token：

1. TopK 选择 192 个位置。
2. 这些 token 通过当前 block 的 Attention、MLP 和两个 MoE Adapter。
3. 选中 token 的输出乘以 `1 + softmax(score)`。
4. 处理后的 token scatter 回原位置；其余 65 个 token 原样旁路。

在首个路由 block，token 分数的软排名还会与“局部 token 对全局 CLS 的余弦相似度排名”计算 Spearman 相关性，优化 `1 - rho`。这让选择器偏向与全局人脸语义一致的局部线索，而不是直接用标签作为 token 输入。

In [ ]:
total_tokens = tokens_per_crop
selected = selected_tokens
bypassed = total_tokens - selected

print(f"selected ratio: {selected / total_tokens:.3%}")
print(f"selected tokens: {selected}")
print(f"bypassed tokens: {bypassed}")
print("sequence width before/after routing:", spec.hidden_dim, "->", spec.hidden_dim)

assert selected == 192
assert selected + bypassed == 257


## 5. Expert Routing：决定 token 如何组合 Adapter

`MoEAdapter` 把 `[B, L, 1024]` 展平为 `[B*L, 1024]`，为每个 token 产生 4 个 gate 权重。每个 expert 是 `1024 -> 256 -> 1024` 的瓶颈 MLP，多个 expert 输出按归一化 gate 权重相加，再通过残差返回原形状。

需要准确理解当前配置：`num_experts=4, top_k=4` 表示每个 token 使用全部四个 expert，只是权重不同；它是“专家混合”，不是计算意义上的稀疏 TopK。`noise=true` 只在训练时扰动 gate。

In [ ]:
active_fraction = spec.top_k / spec.experts
print("experts per token:", spec.top_k)
print(f"expert activation fraction: {active_fraction:.0%}")
print("sparse expert dispatch:", spec.top_k < spec.experts)

assert active_fraction == 1.0


## 6. 训练信号和可训练参数

总损失是 `cross_entropy(logits, label) + 0.1 * loss_spearman`。CLIP 的原始参数全部冻结；可训练部分包括：

- 24 层 attention/MLP 后的 MoE Adapter；
- 最后四层的 LayerRouter 和 TokenRouter；
- 多尺度 query token 与 query attention；
- 1536 -> 2 分类头。

因此这是参数高效迁移：基础视觉表示保持稳定，训练集中在路由、专家和分类。交叉熵、冻结骨干和 Adapter 都是成熟实践；层/token/专家三种选择机制的联合优化是论文核心。

## 7. 当前代码中必须知道的实现细节

这些不是自动判定的 bug，而是复现和创新实验前应明确记录的事实：

1. 论文公式 16 写成对 `m` 个路由层 rank loss 的平均；当前代码只在首个路由层（index 20）传入全局特征并计算 Spearman，后续层仍做 Token Selection，但不累计 rank loss。
2. `load_balancing_weight` 会进入 `MoEAdapter` 配置，但当前损失没有使用它。
3. `top_k=4` 且专家数为 4，所以当前专家混合不稀疏。改为更小 top-k 会改变冻结协议，必须单独建实验分支和做消融。
4. 多尺度 helper 当前固定假设四个局部 crop；若改变输入分辨率或 scale，需要先解除 `.repeat(4, ...)` 的结构假设。

可研究但尚未证明的方向包括：逐层累计 rank loss、真正稀疏专家 + 负载均衡、可学习 token capacity、不同数据域的退出深度校准。每一项都应先做单变量消融，不能直接写成方法优势。

## 8. 回到源码的阅读顺序

1. `training/detectors/utils/core_query_loss.py::forward_query_loss`
2. `training/detectors/dfd_hr_detector.py::features`
3. `training/detectors/dfd_hr_detector.py::features_encoder`
4. `training/detectors/dfd_hr_detector.py::features_encoder_layer_select`
5. `training/detectors/utils/moe_adapter.py::MoEAdapter.forward`
6. `training/detectors/dfd_hr_detector.py::get_losses`

读每个函数时只问三件事：输入形状是什么、谁被选择或旁路、输出形状是否保持。这样可以把论文中的“分层路由”还原成可检查的数据流。